# Retinal OCT Image Classification with DenseNet121 and MONAI

## Overview & Objectives
This notebook builds an end-to-end deep learning pipeline for multi-class classification of Optical Coherence Tomography (OCT) retinal images using PyTorch and MONAI.

### Key Workflow Steps:
1. **Environment Setup & Data Mounting**: Install MONAI and mount Google Drive.
2. **Data Extraction & Exploratory Data Analysis**: Unzip dataset and inspect label distributions and dimensions.
3. **Data Preprocessing & Augmentations**: Define MONAI transforms for intensity scaling, channel positioning, and spatial augmentations.
4. **PyTorch Dataset & DataLoader Construction**: Create custom dataset containers and batch data loaders.
5. **Model Initialization & Training**: Train a 2D DenseNet121 architecture, evaluating ROC-AUC and Accuracy per epoch.
6. **Model Evaluation**: Generate training/validation curves and a comprehensive classification report on holdout test data.

### Prerequisites & Dependencies
- Google Colab environment with GPU runtime enabled.
- External package: 'monai' (installed via pip).

## 1. Setup & Environment  
Mount Google Drive to access stored dataset archives and install/import required dependencies/libraries.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q monai

In [ ]:
import os
import zipfile
import matplotlib.pyplot as plt
import PIL
import torch
import numpy as np

from sklearn.metrics import classification_report
from torch.utils.tensorboard import SummaryWriter
from monai.config import print_config
from monai.data import decollate_batch, DataLoader
from monai.metrics import ROCAUCMetric
from monai.networks.nets import DenseNet121
from monai.transforms import (
    Activations,
    EnsureChannelFirst,
    AsDiscrete,
    Compose,
    LoadImage,
    RandFlip,
    RandRotate,
    RandZoom,
    Resize,
    ScaleIntensity,
)
from monai.utils import set_determinism

print_config()

In [ ]:
set_determinism(seed=0)

## 2. Dataset Extraction & Exploration
Locate the compressed OCT image archive on Google Drive and extract it to local runtime storage.

In [ ]:
zip_path = "/content/drive/MyDrive/ZhangLabData.zip"
extract_dir = "/content/data_dir"

# Unpack archive only if not previously extracted
if not os.path.exists(extract_dir):
  print("Extracting dataset from Google Drive...")
  with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
    print("Done!")

In [ ]:
root_dir = "./"

In [ ]:
data_dir = os.path.join(extract_dir, "CellData", "OCT", "train")

### 2.1 Dataset Categories & Inspect Dimensions
Scan dataset subdirectories to identify target classes, total image counts, and baseline image resolutions.

In [ ]:
# Derive target label names from directory hierarchy
class_names = sorted(x for x in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, x)))
num_class = len(class_names)

# Gather file paths grouped by target class
image_files = [
    [os.path.join(data_dir, class_names[i], x) for x in os.listdir(os.path.join(data_dir, class_names[i]))]
    for i in range(num_class)
]
num_each = [len(image_files[i]) for i in range(num_class)]

# Flatten paths and build numerical class label indices
image_files_list = []
image_class = []
for i in range(num_class):
    image_files_list.extend(image_files[i])
    image_class.extend([i] * num_each[i])
num_total = len(image_class)
image_width, image_height = PIL.Image.open(image_files_list[0]).size

print(f"Total image count: {num_total}")
print(f"Image dimensions: {image_width} x {image_height}")
print(f"Label names: {class_names}")
print(f"Label counts: {num_each}")

### 2.2 Visualizing Random Dataset Samples
Plot a 3x3 grid of randomly sampled OCT scans to inspect image quality, contrast, and category labels.

In [ ]:
plt.subplots(3, 3, figsize=(8, 8))

# Display 9 random samples with category labels
for i, k in enumerate(np.random.randint(num_total, size=9)):
  im = PIL.Image.open(image_files_list[k])
  arr = np.array(im)
  plt.subplot(3, 3, i + 1)
  plt.xlabel(class_names[image_class[k]])
  plt.imshow(arr, cmap="gray", vmin=0, vmax=255)

plt.tight_layout()
plt.show()

## 3. Train / Validation / Test Splitting & Transforms
Partition file paths into 80% training, 10% validation, and 10% test splits using shuffled indices.

In [ ]:
val_frac = 0.1
test_frac = 0.1
length = len(image_files_list)
indices = np.arange(length)
np.random.shuffle(indices)

# Compute index split boundaries
test_split = int(test_frac * length)
val_split = int(val_frac * length) + test_split

test_indices = indices[:test_split]
val_indices = indices[test_split:val_split]
train_indices = indices[val_split:]

# Assign file paths and ground-truth targets to respective subsets
train_x = [image_files_list[i] for i in train_indices]
train_y = [image_class[i] for i in train_indices]
val_x = [image_files_list[i] for i in val_indices]
val_y = [image_class[i] for i in val_indices]
test_x = [image_files_list[i] for i in test_indices]
test_y = [image_class[i] for i in test_indices]

print(f"Training count: {len(train_x)}, Validation count: " f"{len(val_x)}, Test count: {len(test_x)}")

### 3.1 Defining MONAI Preprocessing & Data Augmentation Pipelines
Configure image transforms:
- **Spatial Standardization**: Resize scans to 224x224 input size.
- **Intensity Normalization**: Rescale pixel intensities.
- **Channel Format**: Ensure 1-channel grayscale ordering for PyTorch tensors.
- **Data Augmentations (Training Only)**: Apply random rotations, horizontal flips, and spatial zoom to prevent overfitting.

In [ ]:
IMAGE_SIZE = (224, 224)

# Training pipeline incorporates stochastic spatial transformations
train_transforms = Compose(
    [
        LoadImage(image_only=True),
        EnsureChannelFirst(),
        ScaleIntensity(),
        Resize(IMAGE_SIZE),
        RandRotate(range_x=np.pi / 12, prob=0.5, keep_size=True),
        RandFlip(spatial_axis=1, prob=0.5),
        RandZoom(min_zoom=0.9, max_zoom=1.1, prob=0.5, keep_size=True),
    ]
)

# Validation/testing pipeline uses deterministic preprocessing without augmentation
val_transforms = Compose([LoadImage(image_only=True), EnsureChannelFirst(), ScaleIntensity(), Resize(IMAGE_SIZE)])

# Post-processing transforms for metric evaluation
y_pred_trans = Compose([Activations(softmax=True)])
y_trans = Compose([AsDiscrete(to_onehot=num_class)])

## 4. PyTorch Dataset & DataLoader Construction
Encapsulate image paths and label data in a PyTorch 'Dataset' class, and construct batch 'DataLoader' instances.

In [ ]:
class MedNISTDataset(torch.utils.data.Dataset):
  """Custom Dataset wrapper for loading OCT images on-the-fly with MONAI transforms.

  Args: image_files (list[str]): List of image file paths.
        labels (list[int]): Corresponding integer class labels.
        transforms (callable): MONAI composition of transforms to apply on load.
  """
    def __init__(self, image_files, labels, transforms):
        self.image_files = image_files
        self.labels = labels
        self.transforms = transforms

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        return self.transforms(self.image_files[index]), self.labels[index]

# Instantiating DataLoaders with a batch size of 64 and 2 worker threads
train_ds = MedNISTDataset(train_x, train_y, train_transforms)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)

val_ds = MedNISTDataset(val_x, val_y, val_transforms)
val_loader = DataLoader(val_ds, batch_size=64, num_workers=2)

test_ds = MedNISTDataset(test_x, test_y, val_transforms)
test_loader = DataLoader(test_ds, batch_size=64, num_workers=2)

### 4.1 Model Architecture & Hyperparameter Configuration
Instantiate a 2D DenseNet121 architecture configured for single-channel input, Cross-Entropy loss, and Adam optimization.

In [ ]:
# Select hardware acceleration if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Single input channel for grayscale OCT scans, output layer matching target category count
model = DenseNet121(spatial_dims=2, in_channels=1, out_channels=num_class).to(device)
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), 1e-5)

# Hyperparameters & evaluation metrics
max_epochs = 4
val_interval = 1
auc_metric = ROCAUCMetric()

## 5. Model Training &amp; Validation Epoch Loop
Train the DenseNet121 model over specified epochs, logging batch losses to TensorBoard and checkpointing the best model based on validation ROC-AUC.

In [ ]:
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []
writer = SummaryWriter()

for epoch in range(max_epochs):
    print("-" * 10)
    print(f"epoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0
    # Batch training
    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data[0].to(device), batch_data[1].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        print(f"{step}/{len(train_ds) // train_loader.batch_size}, " f"train_loss: {loss.item():.4f}")
        epoch_len = len(train_ds) // train_loader.batch_size
        writer.add_scalar("train_loss", loss.item(), epoch_len * epoch + step)
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    # Validation
    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad():
            y_pred = torch.tensor([], dtype=torch.float32, device=device)
            y = torch.tensor([], dtype=torch.long, device=device)
            for val_data in val_loader:
                val_images, val_labels = (
                    val_data[0].to(device),
                    val_data[1].to(device),
                )
                y_pred = torch.cat([y_pred, model(val_images)], dim=0)
                y = torch.cat([y, val_labels], dim=0)
            y_onehot = [y_trans(i) for i in decollate_batch(y, detach=False)]
            y_pred_act = [y_pred_trans(i) for i in decollate_batch(y_pred)]
            auc_metric(y_pred_act, y_onehot)
            result = auc_metric.aggregate()
            auc_metric.reset()
            del y_pred_act, y_onehot
            metric_values.append(result)
            acc_value = torch.eq(y_pred.argmax(dim=1), y)
            acc_metric = acc_value.sum().item() / len(acc_value)
            # Save weights when validation ROC-AUC reaches a new peak
            if result > best_metric:
                best_metric = result
                best_metric_epoch = epoch + 1
                torch.save(model.state_dict(), os.path.join(root_dir, "best_metric_model.pth"))
                print("saved new best metric model")
            print(
                f"current epoch: {epoch + 1} current AUC: {result:.4f}"
                f" current accuracy: {acc_metric:.4f}"
                f" best AUC: {best_metric:.4f}"
                f" at epoch: {best_metric_epoch}"
            )
            writer.add_scalar("val_accuracy", acc_metric, epoch + 1)

print(f"train completed, best_metric: {best_metric:.4f} " f"at epoch: {best_metric_epoch}")
writer.close()

## 6. Training Dynamics & Final Test Set Evaluation
Plot loss curves and validation AUC metrics over time, then load the best model checkpoint to evaluate test set performance.

In [ ]:
# Plot epoch loss and validation AUC trajectory
plt.figure("train", (12, 6))

plt.subplot(1, 2, 1)
plt.title("Epoch Average Loss")
x = [i + 1 for i in range(len(epoch_loss_values))]
y = epoch_loss_values
plt.xlabel("epoch")
plt.plot(x, y)
plt.subplot(1, 2, 2)
plt.title("Val AUC")
x = [val_interval * (i + 1) for i in range(len(metric_values))]
y = metric_values
plt.xlabel("epoch")
plt.plot(x, y)
plt.show()

In [ ]:
# Restore weights from the top-performing validation epoch
model.load_state_dict(torch.load(os.path.join(root_dir, "best_metric_model.pth"), weights_only=True))
model.eval()
y_true = []
y_pred = []
with torch.no_grad():
    for test_data in test_loader:
        test_images, test_labels = (
            test_data[0].to(device),
            test_data[1].to(device),
        )
        pred = model(test_images).argmax(dim=1)
        for i in range(len(pred)):
            y_true.append(test_labels[i].item())
            y_pred.append(pred[i].item())

In [ ]:
# Display metrics breakdown across OCT categories
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))